In [0]:
import os
import sys
import importlib


src_path = os.path.abspath("../src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

import silver

importlib.reload(silver)

In [0]:
silver.run_silver(spark)

In [0]:
silver_df = spark.table(silver.SILVER_TABLE)
rejected_df = spark.table(silver.REJECTED_TABLE)

print("Silver schema:")
silver_df.printSchema()

print("Silver sample:")
display(silver_df.limit(10))

print("Rejected sample:")
display(rejected_df.limit(10))

print("Rejection reasons:")
display(
    rejected_df
    .groupBy("rejection_reason")
    .count()
    .orderBy("count", ascending=False)
)

In [0]:
from pyspark.sql import functions as F

missing_reasons = (
    rejected_df
    .filter(
        F.col("rejection_reason").isNull()
        | (F.trim(F.col("rejection_reason")) == "")
    )
    .limit(1)
    .count()
)

if missing_reasons > 0:
    raise ValueError("Rejected records have no rejection reason.")

if not silver.reconcile(spark):
    raise ValueError("Silver reconciliation failed.")

if not silver.check_idempotency(spark):
    raise ValueError("Silver idempotency check failed.")

print("Silver pipeline completed successfully.")